In [1]:
import duckdb
import pandas as pd

In [2]:
#   Analysing a particular scenario
OBSERVATIONS_PATH = "/Users/saranyapal/Developer/tracklet-tracker/data/candidate-packet/data/train/scenario_001/observations.jsonl"
LABELS_PATH = "/Users/saranyapal/Developer/tracklet-tracker/data/candidate-packet/data/train/scenario_001/labels.jsonl"
SUPPORTING_PATH = "/Users/saranyapal/Developer/tracklet-tracker/data/candidate-packet/data/train/scenario_001/supporting.jsonl"

#   JSON
SNAPSHOT_PATH = "/Users/saranyapal/Developer/tracklet-tracker/data/candidate-packet/data/train/scenario_001/snapshot.json"

In [4]:
def process_sql_query(query: str):
    return duckdb.query(query).df()


def deduplicating_df(df: pd.DataFrame, subset: str) -> pd.DataFrame:
    df = df.drop_duplicates(subset=[subset], keep=False)
    
    return df.reset_index(drop=True)


def process_observations(observations_path: str) -> pd.DataFrame:
    observations_df = pd.read_json(observations_path, lines=True)
    observations_df = deduplicating_df(observations_df, "observation_id")

    #   Removing all rows with invalid latitude and longitude
    observations_df = observations_df[
        observations_df["latitude"].between(-90.0, 90.0) &
        observations_df["longitude"].between(-180.0, 180.0)
    ]

    #   Converting to DateTime for splink
    observations_df['observed_at'] = pd.to_datetime(observations_df['observed_at'])
    observations_df['received_at'] = pd.to_datetime(observations_df['received_at'])

    attributes_df = pd.json_normalize(observations_df["attributes"]).add_prefix("observ_")
    observations_df = pd.concat(
        [observations_df.drop(columns=["attributes"]), attributes_df],
        axis=1
    )

    return observations_df




def process_supporting(supporting_path: str) -> pd.DataFrame:
    supporting_df = pd.read_json(supporting_path, lines=True)
    supporting_df = deduplicating_df(supporting_df, "record_id")

    if 'received_at' in supporting_df.columns:
        supporting_df['received_at'] = pd.to_datetime(supporting_df['received_at'])


    if "attributes" in supporting_df.columns:
        attributes_df_2 = pd.json_normalize(supporting_df["attributes"]).add_prefix("supporting_")
        supporting_df = pd.concat(
            [supporting_df.drop(columns=["attributes"]), attributes_df_2],
            axis=1
        )

    return supporting_df




def read_scenario(observations_path: str, labels_path: str, supporting_path: str, snapshot_path: str):
    observations_df = process_observations(observations_path)    
    labels_df = pd.read_json(labels_path, lines=True)
    supporting_df = process_supporting(supporting_path)
    snapshot_df = pd.read_json(snapshot_path)       

    return observations_df, labels_df, supporting_df, snapshot_df





def table_join(observations_df, supporting_df):
    merged_df = pd.merge(
        observations_df,
        supporting_df,
        on="observation_id",
        how="left",
        suffixes=('', '_supp')
    )

    #   Sorting by tracklet_id and time (Probably the most important step)
    merged_df = merged_df.sort_values(by=["tracklet_id", "observed_at"]).reset_index(drop=True)

    return merged_df



def aggregation(merged_df: pd.DataFrame) -> pd.DataFrame:
    #   The clarifying document mentions:
    #   "Each tracklet contains observations from exactly one aircraft. 
    #   Its membership is fixed. You do not need to split tracklets, merge duplicate tracklets, 
    #   or question whether observations within a tracklet belong together."
    agg_funcs = {
        'observed_at': ['first', 'last'],
        'latitude': ['first', 'last'],
        'longitude': ['first', 'last']
    }

    tracklets_summary = merged_df.groupby('tracklet_id').agg(agg_funcs).reset_index()
    tracklets_summary.columns = [
        'tracklet_id', 
        'start_time', 'end_time', 
        'start_lat', 'end_lat', 
        'start_lon', 'end_lon'
    ]

    ## Additional work:
    #       For now, we have not even used the supporting_df's informations. Need to use them for sure
    #       Can use the extra attributes 

    return tracklets_summary

In [5]:
observations_df, labels_df, supporting_df, snapshot_df = read_scenario(
    OBSERVATIONS_PATH,
    LABELS_PATH,
    SUPPORTING_PATH,
    SNAPSHOT_PATH
)

print(observations_df.shape)
print(labels_df.shape)
print(supporting_df.shape)
print(snapshot_df.shape)

(302, 18)
(24, 2)
(180, 8)
(3, 10)


In [8]:
master_df = table_join(observations_df, supporting_df)
aggregated_df = aggregation(master_df)

print(master_df.shape)
print(aggregated_df.shape)

(302, 25)
(24, 7)


In [9]:
master_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 302 entries, 0 to 301
Data columns (total 25 columns):
 #   Column            Non-Null Count  Dtype              
---  ------            --------------  -----              
 0   observation_id    302 non-null    str                
 1   tracklet_id       302 non-null    str                
 2   observed_at       302 non-null    datetime64[us, UTC]
 3   received_at       302 non-null    datetime64[us, UTC]
 4   latitude          302 non-null    float64            
 5   longitude         302 non-null    float64            
 6   source            302 non-null    str                
 7   altitude_ft       239 non-null    float64            
 8   observ_c2         76 non-null     float64            
 9   observ_ref        201 non-null    str                
 10  observ_q          277 non-null    object             
 11  observ_node       302 non-null    str                
 12  observ_r7         302 non-null    float64            
 13  observ_v        

In [ ]:
#   Important Notes:

#   From observations_df:
#       1. unique_observation_count: 312
#       2. observation_row_count: 322
#       3. unique_tracklet_count: 24


#   From supporting_df:
#       1. unique_observation_count: 186

#   From labels_df:
#       1. unique_tracklet_count: 24

## All Observations:

1. [DONE] This indicates that some observations don't have supporting data (meaning a standard left join is the expected behavior.)   

2.  This also indicates that all observations do have their corresponding labels (which is just the successor_tracklet_id: next_tracklet_id) -> This represents the next link   

3. (Common sense) Drop IDs before ML: While you need tracklet_id, observation_id, and scenario_id to join and group the data, you must not use them as predictive features for your ML model.   

4. [DONE] Duplicates must be removed from observations_df and supporting_df!! Before you do the left join, you must deduplicate the supporting.jsonl records using their record_id. (Just like you deduplicate observations using observation_id).   

5. [DONE] If the observation_id is only duplicate, other fields are not -> we must remove both these     
 
6. [DONE] Coordinate Validation: You must validate coordinate ranges. If a latitude is out of bounds (e.g., greater than 90 or less than -90), you must reject that row. There is actually a specific test file mentioned (fixtures/invalid_coordinate.jsonl) designed to check if your code catches this.    

7. Missing Altitude: The document explicitly warns: "Do not silently treat missing values as zero." Altitude can be null. If you fill it with a 0, your model might think the aircraft suddenly dove to sea level. You need to handle it explicitly (e.g., using interpolation, maintaining it as a NaN, or dropping it if it's too sparse).
8. Strict Separation of Time: Ensure you are strictly using observed_at to calculate physical things like distance and speed. received_at is only when the server received the data—you will only need received_at later when you do the "staged demonstration" to simulate delayed data.


In [10]:
aggregated_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 24 entries, 0 to 23
Data columns (total 7 columns):
 #   Column       Non-Null Count  Dtype              
---  ------       --------------  -----              
 0   tracklet_id  24 non-null     str                
 1   start_time   24 non-null     datetime64[us, UTC]
 2   end_time     24 non-null     datetime64[us, UTC]
 3   start_lat    24 non-null     float64            
 4   end_lat      24 non-null     float64            
 5   start_lon    24 non-null     float64            
 6   end_lon      24 non-null     float64            
dtypes: datetime64[us, UTC](2), float64(4), str(1)
memory usage: 2.5 KB


### Feature Preparation:

In [ ]:
# Step 1: Smarter Aggregation (Tracklet Level)
# When you group by tracklet_id, don't just grab the first and last points. Calculate internal physics for the tracklet so you know how the aircraft was behaving before the gap.

# A. Boundary Features (The Basics):
# 1. start_time, end_time, start_lat, end_lat, start_lon, end_lon (using .first() and .last()).
# 2. Altitude Extraction (239/302 non-nulls):
# 3. start_alt and end_alt. (Forward fill or backward fill internally if the exact first/last point is null, otherwise leave as NaN).
# 4. Internal Kinematics (How is it flying?):
# 5. internal_duration: end_time - start_time
# 6. internal_distance: Haversine distance between start and end points.
# 7. internal_speed: internal_distance / internal_duration. (This is much safer to use than observ_spd because you know it's in meters/second. The SOURCE_NOTES.md warns that observ_spd has mixed units across feeds).
# 8. internal_bearing: Calculate the compass bearing (0-360 degrees) from the start coordinate to the end coordinate. This tells you which direction the plane is pointing when it disappears.


# B. Identity Extraction (observ_ref):

# 201 out of 302 rows have observ_ref. The source notes mention this is a "reference-like value" (likely a transponder code).
# Aggregate this by grabbing the mode (most frequent value) for the tracklet.

# Drop the Noise: Drop observ_c2, observ_r7, observ_v, observ_tag, supporting_a1, supporting_b4, supporting_rx. Without documentation, these are dangerous to feed to an ML model because they might just be random sensor hardware IDs.



# Step 2: The Pairwise Features (The Magic for XGBoost)
# Now, you create a new dataframe of Candidate Pairs. For every Tracklet A, pair it with a candidate Tracklet B where B.start_time > A.end_time.

# Here is the exact list of features you should calculate for each (A, B) row to feed into XGBoost:

# Temporal & Spatial Gaps:

# gap_time: B.start_time - A.end_time (in seconds).

# gap_distance: Haversine distance from A.end_lat/lon to B.start_lat/lon.

# Kinematic Plausibility (The most important features):

# required_speed: gap_distance / gap_time. (XGBoost will easily learn that if this is > 300 m/s, it should predict 0).

# speed_change_A: abs(required_speed - A.internal_speed). Does the aircraft have to massively speed up or slow down in the blind spot to reach Tracklet B?

# required_climb_rate: (B.start_alt - A.end_alt) / gap_time. If this requires the plane to climb 10,000 feet in 10 seconds, it's not the same plane.

# Directional Alignment (The "Game Changer"):

# gap_bearing: The compass bearing from A.end_lat/lon to B.start_lat/lon.

# turn_angle_penalty: The absolute angular difference between A.internal_bearing and gap_bearing.

# Why this is genius: If Tracklet A is flying North (0°), and Tracklet B is located 50 miles East (90°) just 2 minutes later, the turn_angle_penalty is 90°. XGBoost will learn that commercial planes don't make instantaneous 90-degree turns. If it points straight at B, the penalty is 0°.

# Categorical / Identity Match:

# ref_match: 1 if A.observ_ref == B.observ_ref (and both are not null), else 0.

# cross_source: 1 if A and B came from different feeds, else 0. (Sometimes planes switch radar feeds as they move).